# Train MFFT-Large on Kaggle
**Multi-Frequency Fusion Transformer — Large Variant (6.30M params)**

## Setup
1. **Settings → Accelerator**: GPU T4 x2 (or P100) — Large needs ≥16GB VRAM
2. **Settings → Internet**: On
3. **Add Input** (attach all 11 datasets):
   - `stable-diffusion`, `places365`, `open-images-v7-dataset`
   - `ntire2026`, `midjourney`, `mfft-real`, `genimage-ai`
   - `faceforensics`, `dfdc-faces-of-the-train-sample`
   - `dall-e3`, `celebdf-v2image-dataset`
4. **Secrets → Add Secret**: `HF_TOKEN` = your HuggingFace write token

## What happens
- Downloads the frozen split manifest from HF (must exist — run `train_mfft_base` first)
- Uses the exact same train/val/test split as all other runs
- Checkpoints go to `runs/<run_id>/...`

In [ ]:
# Cell 1: Clone repo & install deps
import subprocess, sys, shutil
from pathlib import Path

REPO_DIR = Path("/kaggle/working/mfft_repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "https://github.com/MIHMahmudEli/ai-image-detection-research.git", str(REPO_DIR)], check=True)
print(f"Cloned repo to {REPO_DIR}")

sys.path.insert(0, str(REPO_DIR))
subprocess.run(["pip", "install", "-q", "huggingface_hub", "open_clip_torch", "scipy", "scikit-learn", "python-dotenv", "tqdm"], check=False)
print("Deps installed.")

In [ ]:
# Cell 1.5: Pre-flight mount check
!python /kaggle/working/mfft_repo/check_mounts.py

In [ ]:
# Cell 2: Verify GPU & HF token
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "No GPU")

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    print(f"HF Token: {token[:8]}...")
except Exception as e:
    print(f"ERROR: HF_TOKEN not found as Kaggle Secret: {e}")

In [ ]:
# Cell 3: Override model variant to large, then run training
import os
os.environ["MFFT_MODEL_VARIANT"] = "large"

!python /kaggle/working/mfft_repo/kaggle_train_resumable.py

In [ ]:
# Cell 4: Plot training curves
import json, os, csv
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Load run state from results.json (written by training subprocess)
try:
    with open("/kaggle/working/results.json") as f:
        results = json.load(f)
    run_id = results["run_id"]
    print(f"Loaded results for run: {run_id}")
except Exception as e:
    print(f"Could not load results.json: {e}")
    run_id = None

if run_id:
    # Import HF_CHECKPOINT_REPO from pipeline_config (not hardcoded)
    sys.path.insert(0, "/kaggle/working/mfft_repo")
    from pipeline_config import HF_CHECKPOINT_REPO

    try:
        from huggingface_hub import hf_hub_download
        results_path = hf_hub_download(
            repo_id=HF_CHECKPOINT_REPO,
            filename=f"runs/{run_id}/logs/metrics.csv",
            repo_type="model", token=hf_token,
        )
        epochs, train_loss, val_loss, train_acc, val_acc, macro_f1, auc = [], [], [], [], [], [], []
        with open(results_path) as f:
            reader = csv.DictReader(f)
            for row in reader:
                epochs.append(int(row["epoch"]))
                train_loss.append(float(row["train_loss"]))
                val_loss.append(float(row["val_loss"]))
                train_acc.append(float(row["train_accuracy"]))
                val_acc.append(float(row["val_accuracy"]))
                macro_f1.append(float(row["val_macro_f1"]))
                auc.append(float(row["val_auc"]))

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        variant = results.get('model_variant', 'base')
        fig.suptitle(f"MFFT-{variant.title()} Training — {run_id}", fontsize=14)

        axes[0, 0].plot(epochs, train_loss, "b-", label="Train")
        axes[0, 0].plot(epochs, val_loss, "r-", label="Val")
        axes[0, 0].set_title("Loss")
        axes[0, 0].set_xlabel("Epoch")
        axes[0, 0].legend()
        axes[0, 0].grid(True)

        axes[0, 1].plot(epochs, train_acc, "b-", label="Train")
        axes[0, 1].plot(epochs, val_acc, "r-", label="Val")
        axes[0, 1].set_title("Accuracy")
        axes[0, 1].set_xlabel("Epoch")
        axes[0, 1].legend()
        axes[0, 1].grid(True)

        axes[1, 0].plot(epochs, macro_f1, "g-", label="Val Macro-F1")
        axes[1, 0].set_title("Macro-F1")
        axes[1, 0].set_xlabel("Epoch")
        axes[1, 0].legend()
        axes[1, 0].grid(True)

        axes[1, 1].plot(epochs, auc, "m-", label="Val AUC")
        axes[1, 1].set_title("AUC-ROC")
        axes[1, 1].set_xlabel("Epoch")
        axes[1, 1].legend()
        axes[1, 1].grid(True)

        plt.tight_layout()
        plt.savefig("/kaggle/working/training_curves.png", dpi=150)
        plt.show()

        best_idx = macro_f1.index(max(macro_f1))
        print(f"\nBest epoch: {epochs[best_idx]}")
        print(f"  Train acc: {train_acc[best_idx]:.2f}%")
        print(f"  Val acc:   {val_acc[best_idx]:.2f}%")
        print(f"  Val F1:    {macro_f1[best_idx]:.4f}")
        print(f"  Val AUC:   {auc[best_idx]:.4f}")
        print(f"  Best model epoch: {results.get('best_epoch', '?')}")
        print(f"  Status: {results.get('status', '?')}")
    except Exception as e:
        print(f"Could not load metrics: {e}")
        print("Check HF repo for results.")

## Results

- Checkpoints: `https://huggingface.co/MohsinElis/mfft-checkpoints/tree/main/runs/<run_id>`
- Run `list_active_runs.py` locally to compare with other model variants